In [4]:
import sys

sys.path.insert(0, "..")
from runner.utils import (
    allocate_benchmarks,
    create_benchmark_campaign,
    display_speedups,
    fetch_all_partial_results,
    load_benchmark_metadata,
    load_results,
)

In [2]:
# If a util function was modified, use this cell to reload it without having to restart the kernel
%run ../runner/utils.py

## Create benchmark campaign(s)

### 20260225 Run 1h sizes 2, 4, ..., 8
TODO (Daniele): please add the code for this campaign here!
Looks like we ran only scip, highs, gurobi with a 24h timeout on c4-standard-8

### 20260316 Run 3h sizes 2, 4, ..., 10

In [2]:
benchmarks_df = load_benchmark_metadata()
benchs_to_run = benchmarks_df[
    (benchmarks_df["Benchmark"] == "pypsa-de-elec-uc")
    & (benchmarks_df["Instance"].str.endswith("-3h"))
]
len(benchs_to_run)

5

In [ ]:
# Create campaign: 1 instance per VM, latest solvers only

vm_yamls = allocate_benchmarks(
    benchs_to_run,
    "Num. variables",
    len(benchs_to_run),
    machine_type="c4-standard-8",  # NOTE: picked a smaller machine size to save costs!
    timeout_seconds=24 * 60 * 60,  # NOTE: 24h timeout
    solvers="gurobi highs scip",  # NOTE: only running best solvers
    years=[2025],  # latest solvers only, so skip creating older conda envs
)

create_benchmark_campaign(
    "20260316-pypsa-de-elec-3h-2to10",
    "pypsa-de-elec-3h-2to10",
    vm_yamls,
)

Allocated. Estimated runtime: 259.6h
  VM 00: 1 instances, 259.6h
  VM 01: 1 instances, 218.2h
  VM 02: 1 instances, 172.0h
  VM 03: 1 instances, 120.9h
  VM 04: 1 instances, 60.8h
Created directory and files in ../infrastructure/benchmarks/20260316-pypsa-de-elec-3h-2to10
Run this campaign from the infrastructure/ directory using the command:
tofu apply -var-file benchmarks/20260316-pypsa-de-elec-3h-2to10/run.tfvars -state=states/20260316-pypsa-de-elec-3h-2to10.tfstate


## Monitor runs

To view running VMs:
```
gcloud compute instances list | sort | tee /dev/tty | grep benchmark-instance | grep -i running | wc -l
```

To SSH into a running VM and see what's happening:
```
gcloud compute ssh projects/compute-app-427709/zones/us-central1-a/instances/benchmark-instance-more-pypsa-de-sizes-04
tail -f /var/log/startup-script.log
cat /solver-benchmark/results/benchmark_results.csv
```

In [5]:
fetch_all_partial_results()

Cleared ../results/partial-results
There are 5 running VMs. Fetching results from: benchmark-instance-pypsa-de-elec-3h-2to10-04	us-central1-a benchmark-instance-pypsa-de-elec-3h-2to10-02	us-central1-a benchmark-instance-pypsa-de-elec-3h-2to10-00	us-central1-a benchmark-instance-pypsa-de-elec-3h-2to10-01	us-central1-a benchmark-instance-pypsa-de-elec-3h-2to10-03	us-central1-a Done.


## Inspect results

To download results:
```
gsutil -m rsync -r gs://solver-benchmarks/results ./results/gcp-results/
```

In [ ]:
results, variability = load_results(
    ["../results/gcp-results/20260211-all-pypsa-de-sizes/"]
)
benchmarks_df = load_benchmark_metadata()
display_speedups(results, benchmarks_df)